# Part 1: tracking viral loads
## Overview
This notebook presents some exploratory wastewater analysis using data from Zürich, Switzerland.
Data are publicly provided by EAWAG (Swiss Federal Institute of Aquatic Science and Technology) under a CC BY 4.0 license. We are loading them here via the [EpiSewer](https://github.com/adrian-lison/EpiSewer) repository.


In [ ]:
# install.packages("robustbase")
library(ggplot2)
theme_set(theme_minimal())
options(repr.plot.width=8, repr.plot.height=5)

In [ ]:
data_url <- "https://raw.githubusercontent.com/adrian-lison/EpiSewer/main/data/SARS_CoV_2_Zurich.rda"
tmp_file <- tempfile(fileext = ".rda")
download.file(data_url, tmp_file, mode = "wb", quiet = TRUE)

obj_env <- new.env()
loaded_names <- load(tmp_file, envir = obj_env)
data_zurich <- obj_env[[loaded_names[1]]]

names(data_zurich)

## Initial data checks
We first inspect the main time series in the dataset: reported cases and wastewater concentration. This gives a quick sense of trends and data quality over time.

In [ ]:
# Reported cases over time
ggplot() +
  geom_point(aes(x = data_zurich$cases$date, y = data_zurich$cases$cases, color = "Cases"), alpha = 0.7) +
  geom_line(aes(x = data_zurich$cases$date, y = data_zurich$cases$cases, color = "Cases"), linewidth = 0.4) +
  labs(title = "Reported Cases Over Time", x = "Date", y = "Cases", color = NULL)

# Wastewater concentration over time
ggplot() +
  geom_point(aes(x = data_zurich$measurements$date, y = data_zurich$measurements$concentration, color = "Concentration"), alpha = 0.7) +
  geom_line(aes(x = data_zurich$measurements$date, y = data_zurich$measurements$concentration, color = "Concentration"), linewidth = 0.4) +
  labs(title = "Wastewater Concentration Over Time", x = "Date", y = "Concentration", color = NULL)


## Merge time series
Next, we merge cases, flow, and concentration by date so that all variables are aligned in a single table for downstream analysis.

In [ ]:
# Merge by date to create one analysis table
merged_data <- merge(data_zurich$cases, data_zurich$flows, by = "date", all = FALSE)
merged_data <- merge(merged_data, data_zurich$measurements, by = "date", all = FALSE)

Daily observed concentrations can be affected by external factors such as rainfall. To try and mitigate this, we will normalise using flow data (daily inflow at the treatment plant).

In [ ]:
# plot flows
ggplot() +
  geom_point(aes(x = merged_data$date, y = merged_data$flow, color = "Flow"), alpha = 0.7) +
  geom_line(aes(x = merged_data$date, y = merged_data$flow, color = "Flow"), linewidth = 0.4) +
  labs(title = "Wastewater Flow Over Time", x = "Date", y = "Flow", color = NULL)

Check the units:

In [ ]:
data_zurich$units

Loads (gc/day) = concentration (gc/mL) * flow (mL/day)

In [ ]:
# Approximate wastewater load
merged_data$loads <- merged_data$concentration * merged_data$flow #/ median(merged_data$flow, na.rm = TRUE)

## Relate wastewater loads to reported cases

We will visualise first the loads and cases on the same plot, and align them to get a sense of their relationship.

In [ ]:
# We plot loads and cases together to get a sense of their relationship
# The scales are different, we will add a second axis on the right for loads
# to do this we need to rescale loads to be on a similar scale to cases, for visualisation only

rescale_factor <- max(merged_data$cases, na.rm = TRUE) / max(merged_data$loads, na.rm = TRUE)
merged_data$loads_rescaled <- merged_data$loads * rescale_factor
ggplot() +
  geom_point(aes(x = merged_data$date, y = merged_data$cases, color = "Cases"), alpha = 0.7) +
  geom_line(aes(x = merged_data$date, y = merged_data$cases, color = "Cases"), linewidth = 0.4) +
  geom_point(aes(x = merged_data$date, y = merged_data$loads_rescaled, color = "Loads (rescaled)"), alpha = 0.7) +
  geom_line(aes(x = merged_data$date, y = merged_data$loads_rescaled, color = "Loads (rescaled)"), linewidth = 0.4) +
  scale_y_continuous(
    name = "Cases",
    sec.axis = sec_axis(~ . / rescale_factor, name = "Loads")
  ) +
  labs(title = "Reported Cases and Wastewater Loads Over Time", x = "Date", color = NULL)

We see that there seem to be a good correspondence between the two time series. However, it seems that the ratio of loads to cases changes between the two waves of cases (why?). Let's cut the data in two periods and fit a simple regression model to each period to quantify the relationship between loads and cases.

In [ ]:
# Split date to compare periods
merged_data$cutoff <- ifelse(merged_data$date <= as.Date("2022-02-15"),yes = "before", no = "after")

# Linear-scale comparison
ggplot(data = merged_data) +
  geom_point(aes(x = cases, y = loads, color = cutoff), alpha = 0.7) +
  geom_smooth(aes(x = cases, y = loads, color = cutoff),
#   method = robustbase::lmrob,
   method = lm) +
  labs(title = "Cases vs Wastewater Loads", x = "Reported cases", y = "Wastewater Loads", color = "cutoff")


We fit a linear model to summarize the association between wastewater load and reported cases, allowing for a change before and after the cutoff date.

In [ ]:
# model summary: loads as a function of cases and period
# summary(robustbase::lmrob(loads ~ cases * cutoff, data = merged_data))
summary(lm(loads ~ cases * cutoff, data = merged_data))

## Epidemiological dynamics
Daily observations can be noisy. We therefore apply a simple kernel smoother to both case counts and wastewater loads to visualize broader temporal patterns.

In [ ]:
# Smooth wastewater loads over time
ok <- is.finite(merged_data$date) & is.finite(merged_data$loads) # only select rows where both date and loads are not NA
merged_data$loads_smooth <- ksmooth(
  x = merged_data$date[ok],
  y = merged_data$loads[ok],
  kernel = "normal",
  bandwidth = 14,
  x.points = merged_data$date
)$y

# Smooth reported cases over time
ok <- is.finite(merged_data$date) & is.finite(merged_data$cases)
merged_data$cases_smooth <- ksmooth(
  x = merged_data$date[ok],
  y = merged_data$cases[ok], 
  kernel = "normal",
  bandwidth = 14,
  x.points = merged_data$date
)$y

# Plot smoothed loads and cases together
ggplot() +
  geom_line(aes(x = merged_data$date, y = merged_data$cases_smooth, color = "Cases (smoothed)"), linewidth = 0.8) +
  geom_line(aes(x = merged_data$date, y = merged_data$loads_smooth * rescale_factor, color = "Loads (smoothed)"), linewidth = 0.8) +
  geom_point(aes(x = merged_data$date, y = merged_data$cases, color = "Cases (smoothed)"), alpha = 0.5) +
  geom_point(aes(x = merged_data$date, y = merged_data$loads * rescale_factor, color = "Loads (smoothed)"), alpha = 0.5) +
  scale_y_continuous(
    name = "Cases (smoothed)",
    sec.axis = sec_axis(~ . / rescale_factor, name = "Loads (smoothed)")
  ) +
  labs(title = "Smoothed Reported Cases and Wastewater Loads Over Time", x = "Date", color = NULL)  



## Effective reproduction number

The effective reproduction number $R(t)$ is a key epidemiological metric that quantifies the average number of secondary infections generated by an infected individual at time $t$. It provides insights into the transmission dynamics of an infectious disease and helps to assess whether an outbreak is growing ($R(t) > 1$), shrinking ($R(t) < 1$), or stable ($R(t) = 1$).

In a compartmental model, one way to relate the effective reproduction number to the time-varying growth rate $r(t)$ of the number of infected individuals $I(t)$ is as follows:

$$
R(t) = 1 + g r(t)
$$

where $g$ is the average generation time. Here we use $g = 5$ days as a simple assumption. As $r(t)$ can be estimated by

$$
\frac{d I(t)}{dt} = r(t) I(t) \implies r(t) = \frac{1}{I(t)} \frac{d I(t)}{dt} = \frac{d\log(I(t))}{dt}
$$

We can estimate the effective reproduction number from the smoothed trajectories using:

$$
R(t) = 1 + g\frac{d\log(I(t))}{dt}
$$



In [ ]:
# Approximate R from smoothed loads and smoothed cases
merged_data$R_loads <- c(NA, 1 + 5 * diff(log(merged_data$loads_smooth)))
merged_data$R_cases <- c(NA, 1 + 5 * diff(log(merged_data$cases_smooth)))

# Compare both R proxies over time
ggplot(data = merged_data) +
  geom_line(aes(x = date, y = R_loads, color = "R from loads"), linewidth = 0.7) +
  geom_line(aes(x = date, y = R_cases, color = "R from cases"), linewidth = 0.7) +
  geom_hline(yintercept = 1, linetype = "dashed") +
  labs(title = "Approximate Effective Reproduction Number", x = "Date", y = "R(t)", color = NULL)

# Part 2: sequencing data
## Overview
In this part, we will explore the sequencing data. Having information on the circulating variant will add more context to the observed trends in loads and cases, and can help us understand how different variants may have contributed to the observed dynamics. 

In [ ]:
# Load deconvolved variant proportions 
deconvolved <- read.csv("https://github.com/dr-david/tutorial_cambodia_2026/blob/main/deconvolved_zurich.csv?raw=true", stringsAsFactors = FALSE, check.names = FALSE)
deconvolved$date <- as.Date(deconvolved$date)


# Merge with loads
loads_by_variant <- merge(
  deconvolved,
  merged_data[, c("date", "loads")],
  by = "date",
  all.x = TRUE
)

# Variant-attributed loads (proportional split)
loads_by_variant$loads_variant <- loads_by_variant$loads * loads_by_variant$proportion


# Plot loads colored by variant fraction
ggplot(data = loads_by_variant) +
  geom_area(aes(x = date, y = loads_variant, fill = variant), position = "stack") +
  labs(
    title = "Variant-Stratified Wastewater Loads in Zurich",
    x = "Date",
    y = "Load × variant proportion",
    fill = NULL
  )

# Plot variant-stratified loads
ggplot(data = loads_by_variant) +
  geom_line(aes(x = date, y = loads_variant, color = variant)) +
  labs(
    title = "Variant-Stratified Wastewater Loads in Zurich",
    x = "Date",
    y = "Load × variant proportion",
    color = NULL
  )

We see that the second wave in the datset corresponds to the emergence and spread of the BA.2 variant. 

## Selection 

If a variant has a higher fitness (tied to an increase in its reproduction number) than another, it will tend to increase in frequency over time. We can use the observed variant frequencies to estimate the relative selective advantage of the variants.

One simple way to derive this: if we assume that variants $X$ and $Y$ are growing exponentially with growth rates $r_X$ and $r_Y$, then their fraction $f_X$ and $f_Y$ will change over time according to:  
$$
f_X(t) = \frac{X(0)e^{r_X t}}{X(0)e^{r_X t} + Y(0)e^{r_Y t}} \quad\text{and}\quad f_Y(t) = \frac{Y(0)e^{r_Y t}}{X(0)e^{r_X t} + Y(0)e^{r_Y t}}
$$

Substituting $r_X = r_Y + s$ (where $s$ is the selection coefficient) gives:
$$
f_X(t) = \frac{X(0)e^{(r_Y + s) t}}{X(0)e^{(r_Y + s) t} + Y(0)e^{r_Y t}} = \frac{X(0)e^{s t}}{X(0)e^{s t} + Y(0)}  = \frac{1}{1 + \frac{Y(0)}{X(0)}e^{-s t}} = \text{logit}^{-1}\left(s t + \log\frac{X(0)}{Y(0)}\right)
$$

We can therefore estimate the selection coefficient $s$ by fitting a logistic regression model to the observed variant frequencies over time. The rate parameter of the logistic regression will give us an estimate of the selection coefficient $s$. 

In [ ]:
# We will first prepare the data by subsetting to the BA.2 variant, and removing dates where "undetermined" is too high

# find dates where undetermined is above 0.5
undetermined_threshold <- 0.5
undetermined_high_dates <- loads_by_variant$date[loads_by_variant$variant == "undetermined" & loads_by_variant$proportion > undetermined_threshold] 

# subset BA.2 data, excluding dates where undetermined is high
ba2_dataset <- loads_by_variant[loads_by_variant$variant == "BA.2" & !loads_by_variant$date %in% undetermined_high_dates, ]


ggplot(ba2_dataset) +
  geom_line(aes(x = date, y = proportion)) +
  labs(
    title = "Fraction of BA.2 variant in wastewater over time",
    x = "Date",
    y = "BA.2 relative fraction"
  )

We can fit a GLM to this data to estimate $s$

In [ ]:
fit_ba2 <- glm(proportion ~ date, data = ba2_dataset, family = quasibinomial(link = "logit"))

ggplot(ba2_dataset) +
  geom_line(aes(x = date, y = proportion, color="observed")) +
  geom_line(aes(x = date, y = predict(fit_ba2, type = "response"), color="fit")) +
    labs(
        title = "Fraction of BA.2 variant in wastewater over time",
        x = "Date",
        y = "BA.2 relative fraction"
    )

We see that despite its simplicity, this model fits the data quite well. 

We can translate the estimated selection coefficient into an estimate of the relative reproduction number of the variant using the formula:
$$R_X = R_Y e^{s g} \Rightarrow R_X / R_Y = e^{s g}
$$
where $g$ is the expected generation time. We assume here $g = 5$ days. 

In [ ]:
s <- unname(coef(fit_ba2)[2]) # extract coef
ci <- confint(fit_ba2)[2, ] # extract confidence intervals

rr <- exp(c(est = s, lo = ci[1], hi = ci[2]) * 5) # convert into relative R advantage 


cat(
    sprintf("Relative R advantage and .95 confint: %.1f%%  [%.1f%%, %.1f%%]\n",
                    100 * rr[1], 100 * rr[2], 100 * rr[3])
)


## Forecasting

We can use this simple model for forecasting the progression of the BA.2 variant. We will use observations only for the first couple weeks of data, right after the introduction of the new variant.  

In [ ]:
# Refit BA.2 fraction model using data only up to 15 January, then predict forward

variant_cutoff_date <- as.Date("2022-01-15")
ba2_short <- ba2_dataset[ba2_dataset$date <= variant_cutoff_date, ]

# refit the model on the short dataset
fit_ba2_short <- glm(proportion ~ date, data = ba2_short, family = quasibinomial(link = "logit"))

# obtain fitted values and confidence intervals on the logit scale
predicted_short_link <- predict(fit_ba2_short, newdata = ba2_dataset, type = "link", se.fit = TRUE)

# transform predictions and intervals back to fraction scale
predicted_short <- plogis(predicted_short_link$fit)
predicted_short_ci <- data.frame(
  date = ba2_dataset$date,
  fit = predicted_short,
  lower = plogis(predicted_short_link$fit - 1.96 * predicted_short_link$se.fit),
  upper = plogis(predicted_short_link$fit + 1.96 * predicted_short_link$se.fit)
)

ggplot(ba2_dataset) +
  geom_line(aes(x = date, y = proportion, color = "observed")) +
  geom_line(aes(x = date, y = predicted_short, color = "fit + prediction"), linetype = "dashed") +
  geom_ribbon(
    data = predicted_short_ci,
    aes(x = date, ymin = lower, ymax = upper),
    fill = "red", alpha = 0.2, inherit.aes = FALSE
  ) +
  geom_vline(xintercept = variant_cutoff_date, linetype = "dashed", color = "grey") +
  labs(
    title = "Fraction of BA.2 variant in wastewater over time",
    x = "Date",
    y = "BA.2 relative fraction"
  )

Despite a slight overestimation of the variant fitness, the forecast is quite good for limited data. 

In [ ]:
s_short <- unname(coef(fit_ba2_short)[2]) # extract coef
ci_short <- confint(fit_ba2_short)[2, ] # extract confidence intervals

rr_short <- exp(c(est = s_short, lo = ci_short[1], hi = ci_short[2]) * 5) # convert into relative R advantage 


cat(
    sprintf("Relative R advantage and .95 confint: %.1f%%  [%.1f%%, %.1f%%]\n",
                    100 * rr_short[1], 100 * rr_short[2], 100 * rr_short[3])
)


## Compare with clinical sequencing 